# Notebook 4 — EDA (the detailed one)

**Job of this notebook:** understand the training split in depth so we can
decide features and a model in Notebooks 5 and 6.

**Reads:** `data/processed/train.parquet` **only**. The validation and
test sets are not opened here — that would leak information about them
into modeling decisions.

**Writes:** `reports/figures/eda_findings.md` (a short findings summary)
plus any saved charts in `reports/figures/`.

> Take your time here — everything after this notebook depends on what
> gets decided in it.


In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config import TRAIN_PATH, REPORTS_FIGURES, FINDINGS_PATH, LABEL_COL

pd.set_option("display.max_columns", 60)
sns.set_style("whitegrid")

train = pd.read_parquet(TRAIN_PATH)
train.shape


## 1. Data types, shape, memory

In [ ]:
train.info(memory_usage="deep")


In [ ]:
dtype_groups = {
    "numerical": train.select_dtypes(include=[np.number]).columns.tolist(),
    "datetime": train.select_dtypes(include=["datetime64[ns]"]).columns.tolist(),
    "categorical/text/id": train.select_dtypes(include=["object"]).columns.tolist(),
}
for k, v in dtype_groups.items():
    print(f"{k} ({len(v)}): {v}")


## 2. Missing values — how many, where, and does the missing itself mean something

In [ ]:
missing = train.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(2)
missing_summary = pd.DataFrame({"n_missing": missing, "pct_missing": missing_pct})
missing_summary[missing_summary["n_missing"] > 0]


In [ ]:
# Does missingness correlate with the label? e.g. an order with a missing
# delivered-by-carrier date, or missing review score, might itself be a
# signal worth engineering into a feature rather than just imputing away.
for col in missing_summary[missing_summary["n_missing"] > 0].index:
    rate_when_missing = train.loc[train[col].isna(), LABEL_COL].mean()
    rate_when_present = train.loc[train[col].notna(), LABEL_COL].mean()
    print(f"{col:35s} late-rate|missing={rate_when_missing:.2%}  late-rate|present={rate_when_present:.2%}")


## 3. Numerical features: statistics, distributions, skew, outliers

In [ ]:
num_cols = [c for c in dtype_groups["numerical"] if c != LABEL_COL]
train[num_cols].describe().T


In [ ]:
skew = train[num_cols].skew().sort_values(key=abs, ascending=False)
print("Most skewed numerical columns:")
print(skew.head(10))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flat, skew.head(6).index):
    sns.histplot(train[col].dropna(), bins=40, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "numerical_distributions.png", dpi=110)
plt.show()


In [ ]:
# Ranges that make no sense — sanity bounds worth flagging
checks = {
    "total_price": (train["total_price"] < 0).sum() if "total_price" in train else None,
    "total_freight_value": (train["total_freight_value"] < 0).sum() if "total_freight_value" in train else None,
    "product_weight_g": (train["product_weight_g"] <= 0).sum() if "product_weight_g" in train else None,
}
checks


## 4. Categorical features: value counts, cardinality, rare/messy categories

In [ ]:
cat_cols = [c for c in dtype_groups["categorical/text/id"]
            if not c.endswith("_id") and "date" not in c.lower()]

for col in cat_cols:
    print(f"\n{col} — cardinality: {train[col].nunique(dropna=True)}")
    print(train[col].value_counts(dropna=False).head(8))


In [ ]:
# Rare categories: anything under 0.5% of rows — candidates to bucket as "other"
for col in cat_cols:
    vc = train[col].value_counts(normalize=True)
    rare = vc[vc < 0.005]
    if len(rare):
        print(f"{col}: {len(rare)} rare categories covering {rare.sum():.2%} of rows")


## 5. Relations with the label: groupby, cross-tabs, correlations

In [ ]:
for col in cat_cols:
    if train[col].nunique() <= 30:
        rates = train.groupby(col)[LABEL_COL].agg(["mean", "count"]).sort_values("mean", ascending=False)
        print(f"\nLate rate by {col} (top 10 by rate, min 30 orders):")
        print(rates[rates["count"] >= 30].head(10))


In [ ]:
corr = train[num_cols + [LABEL_COL]].corr()[LABEL_COL].sort_values(key=abs, ascending=False)
print("Correlation with label:")
print(corr.drop(LABEL_COL).head(15))

plt.figure(figsize=(6, 8))
corr.drop(LABEL_COL).head(15).plot(kind="barh")
plt.title(f"Correlation with {LABEL_COL}")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "label_correlations.png", dpi=110)
plt.show()


## 6. Dates: purchase timing, delivery times, seasonality, weekday effects

In [ ]:
train["purchase_dow"] = train["order_purchase_timestamp"].dt.dayofweek
train["purchase_month"] = train["order_purchase_timestamp"].dt.month
train["approval_to_carrier_days"] = (
    train["order_delivered_carrier_date"] - train["order_approved_at"]
).dt.total_seconds() / 86400
train["estimated_delivery_window_days"] = (
    train["order_estimated_delivery_date"] - train["order_purchase_timestamp"]
).dt.total_seconds() / 86400

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
train.groupby("purchase_dow")[LABEL_COL].mean().plot(kind="bar", ax=axes[0], title="Late rate by weekday (0=Mon)")
train.groupby("purchase_month")[LABEL_COL].mean().plot(kind="bar", ax=axes[1], title="Late rate by month")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "seasonality.png", dpi=110)
plt.show()


## 7. Geography: customer/seller state, cross-state shipments, ZIP prefix

We don't have precise coordinates joined onto this table yet — that's a
Notebook 5 feature-engineering step (joining `olist_geolocation_dataset`
by ZIP prefix to approximate customer↔seller distance). Here we just look
at what's already available.


In [ ]:
train["cross_state"] = (train["customer_state"] != train["seller_state"]).astype(int)

print("Share of cross-state shipments:", train["cross_state"].mean().round(3))
print()
print("Late rate: same-state vs cross-state")
print(train.groupby("cross_state")[LABEL_COL].mean())

top_states = train["customer_state"].value_counts().head(10).index
state_rates = train[train["customer_state"].isin(top_states)].groupby("customer_state")[LABEL_COL].mean()
state_rates.sort_values(ascending=False).plot(kind="bar", title="Late rate by customer state (top 10 by volume)")
plt.tight_layout()
plt.savefig(REPORTS_FIGURES / "late_rate_by_state.png", dpi=110)
plt.show()


## 8. Findings summary — what this decides for Notebooks 5 & 6

Write this in your own words after running the cells above. A starter
template is saved to `reports/figures/eda_findings.md` below — edit it
in place with your actual numbers before moving on.


In [ ]:
findings = """# EDA findings — Notebook 4

## Class balance
- Late-delivery rate in train: <fill in from Notebook 2/3 output>
- Treated as an imbalanced classification problem: yes / no (fill in)

## Missingness
- Columns with meaningful missingness: <fill in, e.g. review_score>
- Plan: <impute / flag with indicator / drop column>

## Strongest signals found
- Numerical: <e.g. estimated_delivery_window_days, total_freight_value>
- Categorical: <e.g. customer_state, payment_type>
- Geography: cross-state shipments show <higher/lower> late rate
- Seasonality: <e.g. late rate spikes in <month>, likely holiday volume>

## Leakage watch-list (must NOT become features)
- order_delivered_customer_date (defines the label)
- order_delivered_carrier_date (only known after the courier picks up —
  confirm at feature-engineering time whether this is available at the
  point we'd actually need to predict lateness)
- review_score / review dates (created after delivery)

## Decisions for Notebook 5 (feature engineering)
- Features to build: <list>
- Encoding plan for categoricals: <one-hot / target encoding / etc.>
- Scaling needed: <yes/no, which model>

## Decisions for Notebook 6 (modeling)
- Baseline: <e.g. majority-class or logistic regression>
- Metric: <e.g. PR-AUC or F1 on the late class, not accuracy>
"""

with open(FINDINGS_PATH, "w") as f:
    f.write(findings)

print(f"Findings template saved to {FINDINGS_PATH} — edit it with your real numbers.")
